In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = 'Adult'

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.read_csv(f"{ugce_dir}/data/adult.csv")
TARGET_COLUMN = 'target'
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]
datasetX = dataset.drop(columns=[TARGET_COLUMN])

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])
    
    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])
    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.8469648889343843
Number of instances to explain:  2063


In [6]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [ ]:
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: age
count    48842.000000
mean        38.643585
std         13.710510
min         17.000000
25%         28.000000
50%         37.000000
75%         48.000000
max         90.000000
Name: age, dtype: float64

 Feature: workclass
count    46043.000000
mean         4.105727
std          1.143796
min          1.000000
25%          4.000000
50%          4.000000
75%          4.000000
max          8.000000
Name: workclass, dtype: float64

 Feature: education
count    47453.000000
mean        10.589573
std          3.501708
min          1.000000
25%          9.000000
50%         11.000000
75%         12.000000
max         15.000000
Name: education, dtype: float64

 Feature: education-num
count    48842.000000
mean        10.078089
std          2.570973
min          1.000000
25%          9.000000
50%         10.000000
75%         12.000000
max         16.000000
Name: education-num, dtype: float64

 Feature: marital-status
count    42209.000000
mean         3.030278
std          1.176

# UGCE

## Dynamic

# Only Immutability

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'age': 'i',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_immutability = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_immutability.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_immutability, open(f'{results_dir}/results_incremental{strategy}_only_immutability_consts_ONE_constraint.pkl', 'wb'))

In [ ]:
# load results
import os, pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_explainer_immutability = pickle.load(open(f'{results_dir}/results_incrementalfix_population_update_fitness_only_immutability_consts_ONE_constraint.pkl', 'rb'))

In [ ]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_immutability, verbose=True)

Full Time: mean = 53.56, std = 0.64
Generations: mean = 6.01, std = 0.00
Coverage: mean = 22.81, std = 0.00
Proximity Loss: mean = 0.05, std = 0.00
Sparsity: mean = 0.03, std = 0.00
Intermediate Best Distances: mean = 0.03, std = 0.00


(53.555899699529014,
 6.010615711252654,
 22.80871670702179,
 0.048202896285418996,
 0.031247542659432238,
 0.034175858712527436)

# Only Range

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'age': (38, 90)
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints


results_incremental_explainer_ranges = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_ranges.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_ranges, open(f'{results_dir}/results_incremental{strategy}_only_range_consts_ONE_constraint.pkl', 'wb'))

In [ ]:
# load results
import os, pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_explainer_ranges = pickle.load(open(f'{results_dir}/results_incrementalfix_population_update_fitness_only_range_consts_ONE_constraint.pkl', 'rb'))

In [12]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_ranges, verbose=True)

Full Time: mean = 59.93, std = 0.00
Generations: mean = 6.12, std = 0.00
Coverage: mean = 23.34, std = 0.00
Proximity Loss: mean = 0.03, std = 0.00
Sparsity: mean = 0.02, std = 0.00
Intermediate Best Distances: mean = 0.02, std = 0.00


(59.925490856170654,
 6.1203319502074685,
 23.34140435835351,
 0.033939284988934296,
 0.019291724296911106,
 0.021138036685024505)

# Only Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "age": 'incr',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints


results_incremental_explainer_direct = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_direct.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_direct, open(f'{results_dir}/results_incremental{strategy}_only_directionality_consts_ONE_constraint.pkl', 'wb'))

In [7]:
## load results
import os, pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_explainer_direct = pickle.load(open(f'{results_dir}/results_incrementalfix_population_update_fitness_only_directionality_consts_ONE_constraint.pkl', 'rb'))

In [10]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_direct, verbose=True)

Time taken for generating counterfactuals using UGCE dynamic:  1.2808139443397522  minutes
Full Time: mean = 1.28, std = 0.00
Generations: mean = 6.50, std = 0.00
Coverage: mean = 36.85, std = 0.00
Proximity Loss: mean = 0.04, std = 0.00
Sparsity: mean = 0.02, std = 0.00
Intermediate Best Distances: mean = 0.02, std = 0.00


(1.2808139443397522,
 6.504599211563732,
 36.85230024213075,
 0.035394332495069285,
 0.022348153015038768,
 0.018145721074457773)